# 选修E9 Day 3 上机：AI治理框架--从NIST AI RMF到企业安全策略

**版本**：v5.0 学习材料包
**配套**：notes.md（讲义）｜ data/README.md（真实库/数据）｜ solution.ipynb（参考答案，做完再看）

## 学习目标
学完你能：
1. 用 **pydantic** 定义NIST AI RMF控制项schema（Govern/Map/Measure/Manage四函数，18个真实控制项）
2. 实现**三框架风险分级器**：NIST AI RMF（治理方法论）+ EU AI Act（4级风险）+ 中国生成式AI管理办法（备案/标识/透明）对比
3. 用 **pandas** 构建治理台账，追踪**治理闭环**（登记->评估->控制->监控->审计）
4. 设计**企业AI安全策略5层架构**（治理层/评估层/技术防护层/运营层/合规层）
5. 理解 **MCP治理即代码** + **computer use治理风险**（2026前沿，本Day作认知，不实跑）

## 说明
本笔记本有 **6 个 TODO**，你需要自己填写代码。每个 TODO 有提示。
真实库：pydantic（治理schema）+ pandas（治理台账）。
营销映射：为营销Agent系统（推荐/文案/定价/客服/画像/竞品/投放/深合）做NIST合规登记与三框架风险分级。

**关键**：本Day用pydantic治schema + 规则扫描，无需API key。garak/PyRIT/MCP/computer use在notes.md作前沿认知。

## 上机概览

本 notebook 实现企业AI治理工具链：

1. **NIST AI RMF 合规扫描器**：用 pydantic 定义18个真实控制项，对营销AI用例按Govern/Map/Measure/Manage评分
2. **三框架风险分级器**：EU AI Act（4级）+ 中国AI法规（备案/标识/透明）+ NIST（治理成熟度）对比
3. **治理台账 + 闭环追踪**：pandas构建用例清单/风险分级/控制措施/审计记录，追踪登记->评估->控制->监控->审计
4. **企业AI安全策略5层架构**：治理层/评估层/技术防护层/运营层/合规层的落地检查

**6个TODO**：
- TODO1：NIST AI RMF 控制项 Schema（pydantic）
- TODO2：营销AI用例治理登记表（8+用例，三框架分类属性）
- TODO3：NIST AI RMF 合规扫描器
- TODO4：三框架风险分级器（EU AI Act + 中国AI法规 + 框架对比）
- TODO5：治理台账 + 闭环追踪（pandas）
- TODO6：企业AI安全策略5层架构 + 营销AI治理专项分析

In [ ]:
import warnings
warnings.filterwarnings('ignore')

from pydantic import BaseModel, Field
from enum import Enum
from typing import Optional
import pandas as pd

print("pydantic + pandas loaded successfully")
print("NIST AI RMF + EU AI Act + 中国AI法规 三框架治理就绪")
print("本Day用pydantic schema + 规则扫描，无需API key")
print("garak/PyRIT/MCP/computer use 在 notes.md 中作关键词提及（本Day不实跑）")

## TODO1：NIST AI RMF 控制项 Schema（pydantic）

NIST AI RMF 1.0（2023年1月发布）是美国NIST的AI治理框架，核心是四步循环（Govern贯穿全过程）：

| 功能 | 核心问题 | 关键活动 | 营销实践 |
|:----:|---------|---------|---------|
| **Govern** | 谁负责AI治理？ | 治理委员会、使用政策、问责人 | 营销AI使用政策：哪些决策AI可自主/需人审/禁止 |
| **Map** | 有哪些AI系统？风险在哪？ | 用例清单、上下文映射、风险识别 | 梳理营销AI用例（文案/推荐/投放/客服），映射风险 |
| **Measure** | 风险有多大？ | 评估指标、测试、基准对比 | 文案质量评分、推荐公平性、投放偏差监测 |
| **Manage** | 如何应对风险？ | 优先级、缓解措施、持续监控 | 内容安全过滤、偏差阈值告警、人工审核 |

每个控制项有：id（如GOVERN-1）、function、category、description、status、score(0-100)。

**你的任务**：定义 `ComplianceStatus` 枚举、`ControlItem` 模型、`NIST_CONTROL_ITEMS` 列表（18个真实控制项：Govern-5/Map-5/Measure-4/Manage-4）。

In [ ]:
# --- ComplianceStatus 枚举 ---
class ComplianceStatus(str, Enum):
    NOT_ASSESSED = "not_assessed"
    NOT_MET = "not_met"
    PARTIALLY_MET = "partially_met"
    MET = "met"

# --- ControlItem pydantic模型 ---
class ControlItem(BaseModel):
    id: str
    function: str  # Govern / Map / Measure / Manage
    category: str
    description: str
    status: ComplianceStatus = ComplianceStatus.NOT_ASSESSED
    score: float = Field(ge=0, le=100, default=0.0)

# --- NIST AI RMF 1.0 真实控制项（18项，来自官方文档）---
NIST_CONTROL_ITEMS = [
    # Govern（GOVERN-1~5）
    ControlItem(id="GOVERN-1", function="Govern", category="政策与流程",
                description="AI系统的政策、流程、程序和实践已建立并文档化"),
    ControlItem(id="GOVERN-2", function="Govern", category="问责结构",
                description="明确的角色和责任分配，每个AI系统有指定的问责人(Accountable Owner)"),
    ControlItem(id="GOVERN-3", function="Govern", category="人员能力",
                description="团队具备AI风险管理所需的培训、专业知识和资源"),
    ControlItem(id="GOVERN-4", function="Govern", category="利益相关方参与",
                description="外部利益相关方（用户、受影响群体、监管机构）的参与机制已建立"),
    ControlItem(id="GOVERN-5", function="Govern", category="全生命周期治理",
                description="AI系统全生命周期（设计-开发-部署-运维-退役）的治理流程已定义"),
    # Map（MAP-1~5）
    ControlItem(id="MAP-1", function="Map", category="上下文建立",
                description="AI系统的使用上下文已明确记录（业务目标、部署环境、影响范围）"),
    ControlItem(id="MAP-2", function="Map", category="分类与风险识别",
                description="AI系统已按风险等级分类，潜在风险已识别和文档化"),
    ControlItem(id="MAP-3", function="Map", category="能力与限制",
                description="AI系统的能力边界、已知限制和不确定性已文档化"),
    ControlItem(id="MAP-4", function="Map", category="影响评估",
                description="AI系统对个人、群体和社会的潜在影响已评估"),
    ControlItem(id="MAP-5", function="Map", category="第三方风险评估",
                description="第三方数据、模型和组件的来源及风险已评估"),
    # Measure（MEASURE-1~4）
    ControlItem(id="MEASURE-1", function="Measure", category="评估方法选择",
                description="已选择并验证适当的AI风险评估方法（自动测试/人工评估/红队测试）"),
    ControlItem(id="MEASURE-2", function="Measure", category="可信特征评估",
                description="已评估AI系统的准确性、安全性、公平性、隐私性、可解释性等特征"),
    ControlItem(id="MEASURE-3", function="Measure", category="指标追踪",
                description="已建立AI系统性能和风险的持续追踪指标体系"),
    ControlItem(id="MEASURE-4", function="Measure", category="反馈机制",
                description="已建立评估结果的反馈收集和改进机制"),
    # Manage（MANAGE-1~4）
    ControlItem(id="MANAGE-1", function="Manage", category="风险优先级",
                description="已根据风险严重程度和发生概率对风险进行优先级排序"),
    ControlItem(id="MANAGE-2", function="Manage", category="资源分配",
                description="已分配足够的资源处理已识别的高优先级风险"),
    ControlItem(id="MANAGE-3", function="Manage", category="第三方风险处理",
                description="已制定第三方AI组件的风险处理策略（合同/审计/监控）"),
    ControlItem(id="MANAGE-4", function="Manage", category="风险响应",
                description="已建立风险响应流程（消除/降低/转移/接受）和事件应急方案"),
]

print(f"已定义 {len(NIST_CONTROL_ITEMS)} 个NIST AI RMF控制项")
for func in ['Govern', 'Map', 'Measure', 'Manage']:
    items = [c for c in NIST_CONTROL_ITEMS if c.function == func]
    print(f"  {func}: {len(items)}项")

## TODO2：营销AI用例治理登记表

构建营销AI用例治理登记表，包含 **8+个真实营销场景AI用例**。每个用例需标注：
- **EU AI Act分类属性**：用于判断禁止/高风险/有限风险/最小风险
- **中国AI法规属性**：生成式AI备案/算法推荐透明/深度合成标识/个人信息/跨境数据/内容安全
- **NIST评估属性**：人工监督/审计日志/偏见测试/透明度/内容过滤

**营销AI用例清单**（8+个）：
1. AI个性化推荐系统 - 算法推荐
2. AI自动文案生成 - 生成式AI
3. AI动态定价系统 - 数据驱动决策
4. AI客服聊天机器人 - 聊天机器人
5. AI用户画像分析 - 个人信息处理
6. AI竞品分析 - 公开数据
7. AI投放策略优化 - 算法推荐
8. AI深度合成广告 - 深度合成（深伪）
9. AI情感分析定向 - 情感识别（潜在Article 5风险）

**你的任务**：定义 `AIUseCase` 模型（含三框架属性），创建8+个营销AI用例。

In [ ]:
class AIUseCase(BaseModel):
    name: str
    description: str
    # EU AI Act Article 5: 禁止的AI实践
    is_subliminal_manipulation: bool = False
    exploits_vulnerabilities: bool = False
    is_social_scoring: bool = False
    emotion_recognition_workplace: bool = False
    real_time_biometric_id: bool = False
    # EU AI Act Annex III: 高风险AI系统
    is_credit_scoring: bool = False
    is_insurance_pricing: bool = False
    # EU AI Act Article 50: 有限风险（透明度义务）
    is_chatbot: bool = False
    generates_content: bool = False
    is_deepfake: bool = False
    # 中国AI法规属性
    is_generative_ai_service: bool = False      # 生成式AI服务（需备案）
    is_algorithm_recommendation: bool = False   # 算法推荐（需透明/可关闭）
    is_deep_synthesis: bool = False             # 深度合成（需标识）
    involves_personal_info: bool = False        # 涉及个人信息（需知情同意）
    involves_cross_border_data: bool = False    # 跨境数据（需安全评估）
    has_content_safety_risk: bool = False       # 内容安全风险
    # NIST评估属性
    has_human_oversight: bool = False
    has_audit_log: bool = False
    has_bias_testing: bool = False
    has_transparency: bool = False
    has_content_filter: bool = False
    # 治理闭环属性
    owner: str = "未指派"
    control_measure: str = "未定义"
    monitoring_status: str = "未启用"  # 未启用/已启用/告警中
    next_audit_date: str = "未排期"

# 8+ 个真实营销AI用例（覆盖推荐/文案/定价/客服/画像/竞品/投放/深合/情感）
AI_USE_CASES = [
    AIUseCase(
        name="AI个性化推荐系统",
        description="电商平台基于用户行为数据的个性化商品推荐",
        generates_content=True,  # 生成个性化推荐内容 -> EU AI Act 有限风险
        is_algorithm_recommendation=True,
        involves_personal_info=True,
        has_human_oversight=False, has_audit_log=True, has_bias_testing=False,
        has_transparency=True, has_content_filter=True,
        owner="数据团队", control_measure="推荐差异监测+关闭选项",
        monitoring_status="已启用", next_audit_date="2026-10-01",
    ),
    AIUseCase(
        name="AI自动文案生成",
        description="使用LLM自动生成多平台营销文案（微信/小红书/抖音）",
        generates_content=True,
        is_generative_ai_service=True,
        has_content_safety_risk=True,
        has_human_oversight=True, has_audit_log=False, has_bias_testing=False,
        has_transparency=False, has_content_filter=True,
        owner="内容团队", control_measure="人工审核+品牌调性检查",
        monitoring_status="已启用", next_audit_date="2026-09-15",
    ),
    AIUseCase(
        name="AI动态定价系统",
        description="基于用户数据和市场需求动态调整商品价格",
        involves_personal_info=True,
        has_human_oversight=False, has_audit_log=True, has_bias_testing=False,
        has_transparency=False, has_content_filter=False,
        owner="运营团队", control_measure="价格歧视检测+阈值告警",
        monitoring_status="告警中", next_audit_date="2026-08-30",
    ),
    AIUseCase(
        name="AI客服聊天机器人",
        description="面向客户的AI聊天机器人，自动回复客户咨询",
        is_chatbot=True,
        is_generative_ai_service=True,
        involves_personal_info=True,
        has_human_oversight=True, has_audit_log=True, has_bias_testing=False,
        has_transparency=True, has_content_filter=True,
        owner="客服团队", control_measure="人工转接+对话日志",
        monitoring_status="已启用", next_audit_date="2026-10-15",
    ),
    AIUseCase(
        name="AI用户画像分析",
        description="基于用户行为数据构建画像，用于精准定向投放",
        involves_personal_info=True,
        involves_cross_border_data=False,
        has_human_oversight=False, has_audit_log=True, has_bias_testing=False,
        has_transparency=False, has_content_filter=False,
        owner="数据团队", control_measure="最小必要原则+画像透明",
        monitoring_status="已启用", next_audit_date="2026-11-01",
    ),
    AIUseCase(
        name="AI竞品分析",
        description="爬取公开数据自动分析竞品定价、促销、口碑",
        has_content_safety_risk=False,
        has_human_oversight=True, has_audit_log=False, has_bias_testing=False,
        has_transparency=True, has_content_filter=False,
        owner="市场团队", control_measure="数据来源合规审查",
        monitoring_status="未启用", next_audit_date="未排期",
    ),
    AIUseCase(
        name="AI投放策略优化",
        description="基于ROI预测自动调整广告投放渠道和预算分配",
        generates_content=True,  # 生成广告投放策略内容 -> EU AI Act 有限风险
        is_algorithm_recommendation=True,
        involves_personal_info=True,
        has_human_oversight=True, has_audit_log=True, has_bias_testing=False,
        has_transparency=True, has_content_filter=False,
        owner="投放团队", control_measure="预算阈值+人工审批",
        monitoring_status="已启用", next_audit_date="2026-09-30",
    ),
    AIUseCase(
        name="AI深度合成广告",
        description="使用AI生成虚拟主播/产品演示视频用于营销",
        is_deepfake=True,
        is_deep_synthesis=True,
        generates_content=True,
        has_content_safety_risk=True,
        has_human_oversight=True, has_audit_log=False, has_bias_testing=False,
        has_transparency=False, has_content_filter=True,
        owner="创意团队", control_measure="深伪标识+内容审核",
        monitoring_status="已启用", next_audit_date="2026-08-15",
    ),
    AIUseCase(
        name="AI情感分析定向",
        description="分析用户评论情感，用于精准营销定向",
        emotion_recognition_workplace=False,  # 非工作场所，但情感识别仍有风险
        involves_personal_info=True,
        has_human_oversight=False, has_audit_log=False, has_bias_testing=False,
        has_transparency=False, has_content_filter=False,
        owner="数据团队", control_measure="情感标签脱敏+定向透明",
        monitoring_status="未启用", next_audit_date="未排期",
    ),
]

print(f"已注册 {len(AI_USE_CASES)} 个营销AI用例")
for uc in AI_USE_CASES:
    print(f"  {uc.name} ({uc.description[:30]}...)")

## TODO3：NIST AI RMF 合规扫描器

实现合规扫描器：对每个AI用例，逐一评估18个NIST控制项的合规分数（0-100）。

评分逻辑（基于用例的治理属性）：

| 控制功能 | 评分因子 |
|---------|---------|
| Govern | 基础20 + 审计日志20 + 透明度20 + 人工监督20 + 偏见测试20 |
| Map | 基础15 + 透明度25 + 审计日志20 + 偏见测试20 + 人工监督20 |
| Measure | 基础10 + 偏见测试40 + 审计日志25 + 人工监督15 + 透明度10 |
| Manage | 基础15 + 人工监督35 + 审计日志30 + 透明度20 |

**你的任务**：实现 `assess_control()`、`score_to_status()`、`scan_nist_rmf()` 三个函数。

In [ ]:
def assess_control(use_case: AIUseCase, ctrl: ControlItem) -> float:
    """评估单个控制项的合规分数 (0-100)"""
    score = 0.0
    fn = ctrl.function
    if fn == "Govern":
        score += 20  # 基础：政策存在
        if use_case.has_audit_log: score += 20
        if use_case.has_transparency: score += 20
        if use_case.has_human_oversight: score += 20
        if use_case.has_bias_testing: score += 20
    elif fn == "Map":
        score += 15  # 基础：上下文
        if use_case.has_transparency: score += 25
        if use_case.has_audit_log: score += 20
        if use_case.has_bias_testing: score += 20
        if use_case.has_human_oversight: score += 20
    elif fn == "Measure":
        score += 10  # 基础
        if use_case.has_bias_testing: score += 40
        if use_case.has_audit_log: score += 25
        if use_case.has_human_oversight: score += 15
        if use_case.has_transparency: score += 10
    elif fn == "Manage":
        score += 15  # 基础
        if use_case.has_human_oversight: score += 35
        if use_case.has_audit_log: score += 30
        if use_case.has_transparency: score += 20
    return max(0.0, min(100.0, score))

def score_to_status(score: float) -> ComplianceStatus:
    """将分数转为合规状态"""
    if score >= 80: return ComplianceStatus.MET
    elif score >= 50: return ComplianceStatus.PARTIALLY_MET
    elif score > 1: return ComplianceStatus.NOT_MET
    else: return ComplianceStatus.NOT_ASSESSED

def scan_nist_rmf(use_case: AIUseCase) -> list:
    """对AI用例执行NIST AI RMF合规扫描"""
    results = []
    for ctrl in NIST_CONTROL_ITEMS:
        score = assess_control(use_case, ctrl)
        status = score_to_status(score)
        results.append(ctrl.model_copy(update={"score": score, "status": status}))
    return results

# 验证：扫描第一个用例
test_results = scan_nist_rmf(AI_USE_CASES[0])
print(f"用例: {AI_USE_CASES[0].name}")
for func in ['Govern', 'Map', 'Measure', 'Manage']:
    func_items = [r for r in test_results if r.function == func]
    avg = sum(r.score for r in func_items) / len(func_items)
    print(f"  {func}: 平均分={avg:.1f} ({len(func_items)}项)")

## TODO4：三框架风险分级器

实现**三框架风险分级器**：NIST AI RMF（治理方法论）+ EU AI Act（法律合规）+ 中国AI法规（备案/标识/透明）。

### EU AI Act 4级风险（2024年8月生效，2026年分阶段执行）
1. **禁止**（Article 5）：潜意识操纵、社会评分、工作场所情感识别、实时生物识别
2. **高风险**（Annex III）：招聘/信贷/保险/医疗/司法
3. **有限风险**（Article 50）：聊天机器人/AI生成内容/深度伪造 - 透明度义务
4. **最小风险**：无特殊要求

### 中国AI法规体系
| 法规 | 核心要求 | 营销AI影响 |
|------|---------|-----------|
| 《生成式AI服务管理暂行办法》 | 生成式AI服务需备案、内容需安全 | 文案生成工具可能需备案 |
| 《算法推荐管理规定》 | 推荐需透明、可关闭 | 推荐系统需提供关闭选项 |
| 《深度合成管理规定》 | 深度合成内容需标识 | AI图片/视频需标注 |
| 《个人信息保护法》 | 知情同意、最小必要 | 用户画像需获得同意 |

**你的任务**：实现 `classify_eu_ai_act()`、`classify_china_ai_law()`、`compare_three_frameworks()` 三个函数。

In [ ]:
def classify_eu_ai_act(use_case: AIUseCase):
    """按EU AI Act真实条款判定AI用例的风险等级"""
    # --- Article 5: 禁止的AI实践 ---
    prohibited_checks = [
        (use_case.is_subliminal_manipulation, "Article 5: 潜意识操纵"),
        (use_case.exploits_vulnerabilities, "Article 5: 利用特定群体脆弱性"),
        (use_case.is_social_scoring, "Article 5: 社会评分"),
        (use_case.emotion_recognition_workplace, "Article 5: 工作场所/教育机构情感识别"),
        (use_case.real_time_biometric_id, "Article 5: 公共场所实时远程生物特征识别"),
    ]
    for flag, reason in prohibited_checks:
        if flag:
            return ("禁止", reason)
    # --- Annex III: 高风险AI系统 ---
    high_risk_checks = [
        (use_case.is_credit_scoring, "Annex III: 信贷评估和信用评分"),
        (use_case.is_insurance_pricing, "Annex III: 保险定价和风险评估"),
    ]
    for flag, reason in high_risk_checks:
        if flag:
            return ("高风险", reason)
    # --- Article 50: 有限风险（透明度义务）---
    if use_case.is_deepfake:
        return ("有限风险", "Article 50: 深度伪造内容需标注")
    if use_case.is_chatbot:
        return ("有限风险", "Article 50: 聊天机器人需告知用户与AI交互")
    if use_case.generates_content:
        return ("有限风险", "Article 50: AI生成内容需以可检测方式标注")
    # --- 最小风险 ---
    return ("最小风险", "无额外合规要求，建议参照NIST AI RMF自愿性管理")

def classify_china_ai_law(use_case: AIUseCase):
    """按中国AI法规判定合规要求列表"""
    reqs = []
    if use_case.is_generative_ai_service:
        reqs.append(("生成式AI服务备案", "《生成式AI服务管理暂行办法》: 服务上线前需向网信部门备案"))
    if use_case.is_algorithm_recommendation:
        reqs.append(("算法推荐透明义务", "《算法推荐管理规定》: 提供关闭选项+不针对个人特征的选项"))
    if use_case.is_deep_synthesis:
        reqs.append(("深度合成标识义务", "《深度合成管理规定》: AI生成图片/视频/音频需显著标识"))
    if use_case.involves_personal_info:
        reqs.append(("个人信息保护合规", "《个人信息保护法》: 知情同意+最小必要+可撤回"))
    if use_case.involves_cross_border_data:
        reqs.append(("跨境数据安全评估", "《数据安全法》: 跨境传输需通过安全评估"))
    if use_case.has_content_safety_risk:
        reqs.append(("内容安全审核", "《生成式AI服务管理暂行办法》: 生成内容不得违反法律法规和公序良俗"))
    if not reqs:
        reqs.append(("一般合规", "无特殊要求，建议建立内部AI使用规范"))
    return reqs

def compare_three_frameworks(use_case: AIUseCase):
    """三框架对比：NIST治理得分 + EU AI Act分级 + 中国法规要求"""
    # NIST AI RMF 合规扫描
    nist_results = scan_nist_rmf(use_case)
    nist_avg = sum(r.score for r in nist_results) / len(nist_results)
    nist_by_func = {}
    for func in ['Govern', 'Map', 'Measure', 'Manage']:
        items = [r for r in nist_results if r.function == func]
        nist_by_func[func] = sum(r.score for r in items) / len(items)
    # EU AI Act 分级
    eu_level, eu_reason = classify_eu_ai_act(use_case)
    # 中国AI法规
    china_reqs = classify_china_ai_law(use_case)
    return {
        "use_case": use_case.name,
        "nist_avg_score": round(nist_avg, 1),
        "nist_govern": round(nist_by_func["Govern"], 1),
        "nist_map": round(nist_by_func["Map"], 1),
        "nist_measure": round(nist_by_func["Measure"], 1),
        "nist_manage": round(nist_by_func["Manage"], 1),
        "eu_risk_level": eu_level,
        "eu_reason": eu_reason,
        "china_requirements": china_reqs,
        "china_req_count": len(china_reqs),
    }

# 验证：对所有用例执行三框架分级
print("=== 三框架风险分级结果 ===")
for uc in AI_USE_CASES:
    eu_level, eu_reason = classify_eu_ai_act(uc)
    china_reqs = classify_china_ai_law(uc)
    print(f"  {uc.name}: EU={eu_level} | 中国要求={len(china_reqs)}项")

## TODO5：治理台账 + 闭环追踪（pandas）

用 pandas 构建企业AI治理台账，追踪**治理闭环**：

```
登记(Register) -> 评估(Assess) -> 控制(Control) -> 监控(Monitor) -> 审计(Audit)
     |__________________________ 循环 __________________________|
```

| 闭环阶段 | 关键产出 | 本Day实现 |
|---------|---------|----------|
| 登记 | 用例清单 + 责任人 | AIUseCase注册表 |
| 评估 | NIST得分 + 风险分级 | 三框架分级结果 |
| 控制 | 缓解措施 + 责任人 | control_measure字段 |
| 监控 | 监控指标 + 频率 | monitoring_status |
| 审计 | 审计记录 + 下次审计日 | audit_status + next_audit_date |

**你的任务**：实现 `build_governance_ledger()`、`closed_loop_status()` 两个函数，返回治理台账DataFrame。

In [ ]:
def closed_loop_status(use_case: AIUseCase) -> str:
    """判定治理闭环状态：登记->评估->控制->监控->审计"""
    has_control = use_case.control_measure != "未定义"
    has_monitoring = use_case.monitoring_status in ("已启用", "告警中")
    has_audit = use_case.next_audit_date != "未排期"
    if has_control and has_monitoring and has_audit:
        return "完整闭环"
    elif has_control:
        return "部分闭环"
    else:
        return "断链"

def build_governance_ledger(use_cases: list) -> "pd.DataFrame":
    """构建企业AI治理台账（pandas DataFrame）"""
    rows = []
    for uc in use_cases:
        eu_level, eu_reason = classify_eu_ai_act(uc)
        nist_results = scan_nist_rmf(uc)
        nist_avg = sum(r.score for r in nist_results) / len(nist_results)
        china_reqs = classify_china_ai_law(uc)
        # 找最弱控制项
        weakest = min(nist_results, key=lambda c: c.score)
        rows.append({
            "use_case": uc.name,
            "owner": uc.owner,
            "eu_risk_level": eu_level,
            "china_req_count": len(china_reqs),
            "nist_avg_score": round(nist_avg, 1),
            "weakest_control": f"{weakest.id}({weakest.score:.0f})",
            "control_measure": uc.control_measure,
            "monitoring_status": uc.monitoring_status,
            "next_audit_date": uc.next_audit_date,
            "closed_loop": closed_loop_status(uc),
        })
    return pd.DataFrame(rows)

df_ledger = build_governance_ledger(AI_USE_CASES)
print("=== 企业AI治理台账 ===")
print(df_ledger.to_string(index=False))
print(f"\n闭环状态分布:")
print(df_ledger['closed_loop'].value_counts().to_string())

## TODO6：企业AI安全策略5层架构 + 营销AI治理专项分析

### 企业AI安全策略5层架构（综合NIST + EU AI Act + 中国法规）

| 层 | 名称 | 关键控制 | 本Day检查 |
|:--:|------|---------|----------|
| L1 | 治理层 | AI治理委员会、使用政策、RACI、事件响应SOP | TODO6检查 |
| L2 | 评估层 | 风险分级、上线前评估、偏见审计、DPIA | TODO6检查 |
| L3 | 技术防护层 | Prompt Injection防御、输出过滤、权限管理、监控告警 | Day 2已实现 |
| L4 | 运营层 | 持续监控、定期红队、事件响应、培训、合规报告 | TODO6检查 |
| L5 | 合规层 | NIST对标、EU AI Act、中国法规、行业合规 | TODO4已实现 |

**你的任务**：实现 `enterprise_security_check()`（5层架构检查）和 `marketing_governance_analysis()`（营销AI治理专项分析）。

### 2026前沿：MCP治理即代码 + computer use治理风险
- **MCP**（Model Context Protocol）：合规检查Agent + 审计日志MCP Server，治理从"独立流程"变"内嵌能力"
- **computer use**：AI操作浏览器/桌面应用，权限边界模糊、操作不可逆、UI操作难审计 - 需升级MANAGE-4
- **garak/PyRIT**：红队工具系统化发现漏洞（Day 2已认知），本Day治理框架是红队测试的"对标框架"

**注意**：AI治理是"持续过程"非"一次性合规检查"。对应因果阶梯L1（关联分析），生产期需MCP治理即代码 + 在线监控 + 应急响应。

In [ ]:
def enterprise_security_check(use_cases: list) -> "pd.DataFrame":
    """企业AI安全策略5层架构检查"""
    rows = []
    for uc in use_cases:
        nist_results = scan_nist_rmf(uc)
        nist_avg = sum(r.score for r in nist_results) / len(nist_results)
        eu_level, _ = classify_eu_ai_act(uc)
        china_reqs = classify_china_ai_law(uc)
        # L1 治理层：有owner + 有control_measure
        l1 = (uc.owner != "未指派") and (uc.control_measure != "未定义")
        # L2 评估层：NIST已扫描(本Day) + 有偏见测试
        l2 = (nist_avg > 0) and uc.has_bias_testing
        # L3 技术防护层：有content_filter + 有人工监督
        l3 = uc.has_content_filter and uc.has_human_oversight
        # L4 运营层：监控已启用 + 有审计日
        l4 = (uc.monitoring_status in ("已启用", "告警中")) and (uc.next_audit_date != "未排期")
        # L5 合规层：EU分级 + 中国要求已识别
        l5 = (eu_level != "最小风险") or (len(china_reqs) > 1)
        passed = sum([l1, l2, l3, l4, l5])
        rows.append({
            "use_case": uc.name,
            "L1治理": "PASS" if l1 else "FAIL",
            "L2评估": "PASS" if l2 else "FAIL",
            "L3防护": "PASS" if l3 else "FAIL",
            "L4运营": "PASS" if l4 else "FAIL",
            "L5合规": "PASS" if l5 else "FAIL",
            "通过率": f"{passed}/5",
            "nist_avg": round(nist_avg, 1),
        })
    return pd.DataFrame(rows)

def marketing_governance_analysis(use_cases: list) -> "pd.DataFrame":
    """营销AI治理专项分析：三框架综合 + 改进建议"""
    results = []
    for uc in use_cases:
        comparison = compare_three_frameworks(uc)
        nist_results = scan_nist_rmf(uc)
        weakest = min(nist_results, key=lambda c: c.score)
        gap = max(0, 80 - comparison["nist_avg_score"])
        # 改进建议
        if not uc.has_bias_testing:
            suggestion = "优先补齐偏见测试（Measure维度+40分）"
        elif not uc.has_audit_log:
            suggestion = "建立审计日志（Govern/Manage维度+20-30分）"
        elif not uc.has_transparency:
            suggestion = "增加透明度（用户告知+AI标注）"
        elif not uc.has_human_oversight:
            suggestion = "增加人工监督（Manage维度+35分）"
        elif comparison["eu_risk_level"] == "有限风险":
            suggestion = "确保AI生成内容标注（EU AI Act Article 50）"
        else:
            suggestion = "维持现有治理水平，定期复审"
        results.append({
            "use_case": uc.name,
            "EU风险": comparison["eu_risk_level"],
            "中国要求": comparison["china_req_count"],
            "NIST均分": comparison["nist_avg_score"],
            "最弱项": f"{weakest.id}({weakest.score:.0f})",
            "差距": f"{gap:.0f}",
            "改进建议": suggestion,
        })
    return pd.DataFrame(results)

df_security = enterprise_security_check(AI_USE_CASES)
print("=== 企业AI安全策略5层架构检查 ===")
print(df_security.to_string(index=False))

df_mkt = marketing_governance_analysis(AI_USE_CASES)
print("\n=== 营销AI治理专项分析 ===")
print(df_mkt.to_string(index=False))

# 三框架风险分级分布
print("\n=== EU AI Act 风险分级分布 ===")
eu_levels = [classify_eu_ai_act(uc)[0] for uc in AI_USE_CASES]
dist = pd.Series(eu_levels).value_counts()
for level, count in dist.items():
    pct = count / len(AI_USE_CASES) * 100
    print(f"  {level}: {count}个 ({pct:.1f}%)")

# 闭环状态汇总
print("\n=== 治理闭环状态汇总 ===")
loop_status = [closed_loop_status(uc) for uc in AI_USE_CASES]
loop_dist = pd.Series(loop_status).value_counts()
for status, count in loop_dist.items():
    pct = count / len(AI_USE_CASES) * 100
    print(f"  {status}: {count}个 ({pct:.1f}%)")

## 总结

本练习实现了企业AI治理工具链：

1. **NIST AI RMF 合规扫描器**：pydantic定义18个真实控制项，对营销AI用例按Govern/Map/Measure/Manage评分
2. **三框架风险分级器**：EU AI Act（4级法律合规）+ 中国AI法规（备案/标识/透明）+ NIST（治理成熟度）对比
3. **治理台账 + 闭环追踪**：pandas构建用例清单/风险分级/控制措施/审计记录，追踪登记->评估->控制->监控->审计
4. **企业AI安全策略5层架构**：治理层/评估层/技术防护层/运营层/合规层落地检查

**关键发现**：
- 营销AI系统多属于"有限风险"（需透明度标注），但NIST合规得分差异大
- 深度合成广告 + 情感分析定向是高风险/禁止边界，需重点治理
- 治理闭环中"评估->控制"是最易断链的环节（评估发现问题但未落实控制）

**下一步**：结合Day 2的5层Prompt Injection防御，你已具备"技术防护 + 治理框架"双维度AI安全能力。生产环境应用MCP治理即代码 + garak/PyRIT红队 + computer use权限矩阵。

参考 [NIST AI RMF](https://www.nist.gov/itl/ai-risk-management-framework) + [EU AI Act](https://artificialintelligenceact.eu/) + [garak](https://github.com/NVIDIA/garak) + [PyRIT](https://github.com/Azure/PyRIT)。